# GWR, MGWR & Gravity Models

This notebook covers non-parametric spatial regression and origin-destination flow models:

1. **GWR** — Geographically Weighted Regression with adaptive/fixed bandwidth
2. **MGWR** — Multiscale GWR with variable-specific bandwidths
3. **PPML Gravity** — Poisson pseudo-maximum likelihood for trade/flow data
4. **Negative Binomial Gravity** — handling overdispersion

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Random, LinearAlgebra, Statistics

---
## Part 1: Geographically Weighted Regression

GWR relaxes the assumption that regression coefficients are constant across space.
Instead, each location $i$ gets its own coefficients $\beta_i$ estimated from a
distance-weighted local regression:

$$\hat{\beta}_i = (X^\top W_i X)^{-1} X^\top W_i y$$

where $W_i = \text{diag}(K(d_{ij}/h))$ is a diagonal weight matrix using a spatial kernel.

### 1.1 Generate Spatially Varying Data

We create data where:
- **temperature** has a coefficient that increases with the x-coordinate (spatial non-stationarity)
- **rainfall** has a constant coefficient everywhere (global stationarity)

GWR should detect this: `temperature` needs a small bandwidth (local variation),
`rainfall` can use a large bandwidth.

In [ ]:
rng = MersenneTwister(77)
N = 300
coords = hcat(rand(rng, N) .* 10, rand(rng, N) .* 10)
X = randn(rng, N, 2)

# Spatially varying coefficients
β1_true = 1.0 .+ 0.4 .* coords[:, 1]    # varies: 1.0 to 5.0
β2_true = fill(-0.5, N)                   # constant
β0_true = 2.0 .+ 0.2 .* coords[:, 2]     # intercept varies with y-coord

y = β0_true .+ β1_true .* X[:, 1] .+ β2_true .* X[:, 2] .+ 0.5 .* randn(rng, N)

println("True β₁ range: [$(round(minimum(β1_true);digits=1)), $(round(maximum(β1_true);digits=1))]")
println("True β₂: constant at -0.5")

### 1.2 GWR with Adaptive Bandwidth

In [ ]:
m_adapt = fit_gwr(X, y, coords;
    bandwidth = AdaptiveBandwidth(50),
    kernel = BisquareKernel(),
    varnames = ["temperature", "rainfall"])
println(m_adapt)

In [ ]:
# Verify: temperature β should vary more than rainfall β
println("Coefficient spatial variation (SD):")
println("  temperature: $(round(std(m_adapt.coef_local[:, 2]); digits=3)) — should be large")
println("  rainfall:    $(round(std(m_adapt.coef_local[:, 3]); digits=3)) — should be small")

# Correlation between true and estimated local β₁
r = cor(β1_true, m_adapt.coef_local[:, 2])
println("\nCorrelation(true β₁, estimated β₁) = $(round(r; digits=3))")

### 1.3 GWR with Fixed Bandwidth and Different Kernels

In [ ]:
m_gauss = fit_gwr(X, y, coords;
    bandwidth = FixedBandwidth(3.0),
    kernel = GaussianKernel(),
    varnames = ["temperature", "rainfall"])

m_exp = fit_gwr(X, y, coords;
    bandwidth = FixedBandwidth(3.0),
    kernel = ExponentialKernel(),
    varnames = ["temperature", "rainfall"])

println("Kernel comparison (fixed h=3.0):")
println("  Bisquare:    R²=$(round(m_adapt.R2; digits=4))  AICc=$(round(m_adapt.AICc; digits=1))")
println("  Gaussian:    R²=$(round(m_gauss.R2; digits=4))  AICc=$(round(m_gauss.AICc; digits=1))")
println("  Exponential: R²=$(round(m_exp.R2; digits=4))  AICc=$(round(m_exp.AICc; digits=1))")

### 1.4 Automatic Bandwidth Selection (AICc)

Pass `bandwidth=nothing` (default) and GWR will optimize the adaptive bandwidth
via golden-section search minimizing AICc.

In [ ]:
@time m_auto = fit_gwr(X, y, coords; varnames=["temperature", "rainfall"])
println(m_auto)
println("Optimal bandwidth: k=$(round(Int, m_auto.bandwidth)) nearest neighbors")

### 1.5 Multiscale GWR (MGWR)

MGWR allows each variable to have its own bandwidth. A backfitting algorithm
iteratively optimizes variable-specific bandwidths.

**Expected result**: `temperature` gets a small bandwidth (local process),
`rainfall` gets a large bandwidth (global process).

In [ ]:
@time m_mgwr = fit_mgwr(X, y, coords; varnames=["temperature", "rainfall"])
println(m_mgwr)

println("\nVariable-specific bandwidths:")
for (name, bw) in zip(m_mgwr.coefnames, m_mgwr.bandwidths)
    println("  $name: k=$(round(Int, bw))")
end

---
## Part 2: Gravity / Flow Models

Gravity models estimate bilateral flows (trade, migration, commuting) as a function
of origin/destination characteristics and bilateral resistance (distance).

**PPML** (Santos Silva & Tenreyro 2006) is the standard estimator — it handles zeros
naturally and is robust to heteroskedasticity.

### 2.1 Generate Trade Flow Data

In [ ]:
rng2 = MersenneTwister(2024)
N_orig = 20; N_dest = 15; N_obs = N_orig * N_dest

origin_id = repeat(1:N_orig, inner=N_dest)
dest_id   = repeat(1:N_dest, outer=N_orig)

log_distance = rand(rng2, N_obs) .* 4 .+ 2
common_lang  = Float64.(rand(rng2, N_obs) .< 0.2)
X_grav = hcat(log_distance, common_lang)

# True: distance elasticity = -1.3, language bonus = +0.6
β_grav = [-1.3, 0.6]
α_o = randn(rng2, N_orig) .* 0.8     # origin FE (GDP)
α_d = randn(rng2, N_dest) .* 0.6     # destination FE

η = X_grav * β_grav .+ 3.0
for n in 1:N_obs
    η[n] += α_o[origin_id[n]] + α_d[dest_id[n]]
end
μ_true = exp.(η)
flows = Float64[max(0.0, round(μ_true[n] + sqrt(μ_true[n]) * randn(rng2))) for n in 1:N_obs]
flows = max.(flows, 0.0)

n_zeros = count(flows .== 0)
println("$N_orig origins × $N_dest destinations = $N_obs pairs")
println("Zero flows: $n_zeros ($(round(100*n_zeros/N_obs; digits=1))%)")
println("Mean flow: $(round(mean(flows); digits=1)), Max: $(round(maximum(flows); digits=0))")

### 2.2 PPML Gravity with Bilateral Fixed Effects

In [ ]:
m_ppml = fit_gravity(PoissonGravity(), flows, X_grav, origin_id, dest_id;
    fe = BilateralFE(),
    varnames = ["log_distance", "common_language"],
    maxiter = 200)
println(m_ppml)

println("Parameter recovery:")
println("  distance: true=$(β_grav[1]) → est=$(round(m_ppml.coef[1]; digits=3))")
println("  language: true=$(β_grav[2]) → est=$(round(m_ppml.coef[2]; digits=3))")

### 2.3 Multilateral Resistance Terms

The estimated origin and destination fixed effects proxy for Anderson & van Wincoop (2003)
multilateral resistance — the average trade barrier each country faces.

In [ ]:
mr = multilateral_resistance(m_ppml)
println("Outward MR (origin FE) — top 5:")
top5_o = sortperm(mr.outward; rev=true)[1:5]
for i in top5_o
    println("  Origin $i: $(round(mr.outward[i]; digits=3))")
end

println("\nInward MR (destination FE) — top 5:")
top5_d = sortperm(mr.inward; rev=true)[1:5]
for j in top5_d
    println("  Dest $j: $(round(mr.inward[j]; digits=3))")
end

### 2.4 Negative Binomial Gravity (Overdispersion)

In [ ]:
# Add overdispersion
flows_od = Float64[max(0.0, round(μ_true[n] + 2*sqrt(μ_true[n]) * randn(rng2))) for n in 1:N_obs]
flows_od = max.(flows_od, 0.0)

m_nb = fit_gravity(NegBinGravity(), flows_od, X_grav, origin_id, dest_id;
    fe = BilateralFE(),
    varnames = ["log_distance", "common_language"])
println(m_nb)
println("Overdispersion α = $(round(m_nb.alpha; digits=4)) (0 = Poisson, larger = more overdispersion)")

### 2.5 Different Fixed Effects Specifications

In [ ]:
m_orig = fit_gravity(PoissonGravity(), flows, X_grav, origin_id, dest_id;
    fe = OriginFE(), varnames=["log_distance", "common_language"], maxiter=200)

m_dest = fit_gravity(PoissonGravity(), flows, X_grav, origin_id, dest_id;
    fe = DestinationFE(), varnames=["log_distance", "common_language"], maxiter=200)

println("FE specification comparison:")
println("═══════════════════════════════════════════════")
println("                  Bilateral   Origin    Dest")
println("───────────────────────────────────────────────")
println("log_distance    $(round(m_ppml.coef[1];digits=3))     $(round(m_orig.coef[1];digits=3))    $(round(m_dest.coef[1];digits=3))")
println("common_lang     $(round(m_ppml.coef[2];digits=3))      $(round(m_orig.coef[2];digits=3))     $(round(m_dest.coef[2];digits=3))")
println("Pseudo-R²       $(round(m_ppml.pseudo_R2;digits=3))      $(round(m_orig.pseudo_R2;digits=3))     $(round(m_dest.pseudo_R2;digits=3))")
println("═══════════════════════════════════════════════")

### 2.6 Formula Interface for Gravity

In [ ]:
tbl_grav = (flows=flows, log_dist=log_distance, common_lang=common_lang,
            exporter=origin_id, importer=dest_id)

m_form = fit_gravity(PoissonGravity(),
    @formula(flows ~ log_dist + common_lang),
    tbl_grav; origin=:exporter, dest=:importer, maxiter=200)

println("Formula interface result:")
println("  log_dist coef: $(round(m_form.coef[1]; digits=4))")
println("  Same as matrix: $(round(m_ppml.coef[1]; digits=4))")

## Summary

| Model | Function | When to Use |
|-------|----------|-------------|
| GWR | `fit_gwr` | Coefficients vary across space |
| MGWR | `fit_mgwr` | Each variable has different spatial scale |
| PPML Gravity | `fit_gravity(PoissonGravity(), ...)` | O-D flows, zero-inflated, heteroskedastic |
| NB Gravity | `fit_gravity(NegBinGravity(), ...)` | O-D flows with overdispersion |

**Kernels**: `BisquareKernel()`, `GaussianKernel()`, `ExponentialKernel()`

**Bandwidths**: `AdaptiveBandwidth(k)`, `FixedBandwidth(h)`, or `nothing` for AICc auto-selection

**Gravity FE**: `BilateralFE()`, `OriginFE()`, `DestinationFE()`